In [ ]:
import pandas as pd
import numpy as np

In [ ]:
s = pd.Series([1,2,3,np.nan,None,pd.NA])
df = pd.DataFrame([[1,2,3,pd.NA],[4,5,6,None],[7,8,9,np.nan]],columns=['a','b','c','d'],index=['x','y','z'])
#np.nan — 是个 float，不是整数也不是字符串。所以一旦你把它放进整数列，整列会被自动提升成 float64，哪怕其他值都是整数
#None — Python 原生空值，类型是 NoneType。放进 pandas 时行为分裂：如果是字符串列它就保持 None；如果是数值列它会被隐式转成 np.nan，结果和上面一样，列变 float
#pd.NA — pandas 1.0 之后才有的自家缺失值。最大优势：支持可空整数类型 Int64（注意大写）。你终于可以在有缺失值的情况下保持整数列不动
print(s)
print(df)

## 检测缺失值

```python
df.isnull()        # True=缺失，False=非空 → 布尔 DataFrame
df.isnull().sum()  # 每列缺失个数（最常用）
df.isnull().sum(axis=1)   # 每行缺失个数

df.notnull()       # isna 的反向，同上
df.info()          # 一眼看到每列非空数和 dtype
```

- `isna()` 和 `isnull()` 是同一个函数，没有区别
- `np.nan` 是 float 类型 → 放到整数列会强制整列变 `float64`
- `pd.NA` + `dtype='Int64'` 可以让整数列在有空值的情况下保持整数


In [ ]:
print(s.isna())
print(s.isnull()) #二者完全等价

print(df.isna())
print(df.isnull())

#查看个数
print(s.isna().sum())
print(df.isnull().sum()) #默认按每一列查看缺失值个数
print(df.isna().sum(axis=1))

## 处理缺失值

### 删除缺失行/列
```python
df.dropna()                        # 默认 how='any'：有一个 NaN 就删该行
df.dropna(how='all')               # 整行全空才删
df.dropna(thresh=3)                # 至少 3 个非空值才保留
df.dropna(subset=['列A', '列B'])    # 只检查这两列
df.dropna(axis=1)                  # 删列而非删行
```

### 填充缺失值
```python
df.fillna(0)                       # 统一填一个值
df.fillna({'A': 0, 'B': 99})       # 每列不同值
df.fillna(method='ffill')          # 前向填充（抄上面的值）
df.fillna(method='bfill')          # 后向填充（抄下面的值）
df.fillna(df.mean())               # 用列均值填充

# ffill/bfill 也可直接调用
df.ffill()
df.bfill()
```

| 策略 | 适用场景 |
|---|---|
| `fillna(0)` | 计数类数据（缺失即没有） |
| `ffill()` | 时间序列（上一条延续） |
| `fillna(mean())` | 正态分布的数值列 |
| `fillna(median())` | 有离群值的数值列 |
| `dropna()` | 缺失少且不影响分析 |


In [ ]:
#去除
print(s.dropna())
print(df.dropna()) #把有缺失值的行全部删除 默认how='any'
print(df.dropna(how='all')) #如果行所有值都是缺失值才删除
print(df.dropna(thresh=3)) #至少有n个不是缺失值就保留
print(df.dropna(thresh=4))
print('\n')

#接收axis=1则按列删除
print(df.dropna(axis=1))
print(df.dropna(how='all',axis=1))
print(df.dropna(thresh=2,axis=1))
#聚合（sum / mean）：axis = 被压扁的方向
#删除（dropna / drop）：axis = 被删除的东西本身

print(df.dropna(subset=['a','c','d'])) #如果某些列有缺失值，则删除对应行
print(df.dropna(subset=['x','z'],axis=1,thresh=1)) #检查x，z两行，一列中这两行得元素有一个非空就不删
#subset里的标签必须属于你不删的那个轴

In [ ]:
#填充
df = pd.read_csv(r'D:/dsml/data/weather.csv')
print(df.isna().sum(axis=1))
print(df.isna().sum(axis=0))
print(df)

#字典填充
print(df.fillna({'风速(m/s)':10,'湿度(%)':df['湿度(%)'].mean().round(1)})) #不改变原本df
print(df.isna().sum())

#根据附近值填充
#ffill用前一个值，bfill用后一个值
print(df[['温度(℃)','降水量(mm)']].ffill()) #可以接收axis=1
print(df.isna().sum())
